# Chapter 11: File Handling and Exceptions

### Why This Matters

Data lives in files — CSVs, JSON exports from APIs, text logs — and any real analysis pipeline starts by reading them and, often, ends by writing results back out. Meanwhile, files are also the single most common source of runtime errors in a data pipeline (missing file, wrong permissions, malformed data) — so file handling and exception handling are taught together for good reason.

### 11.1 Reading and Writing Text Files

In [ ]:
with open("sample_report.txt", "w") as f:
    f.write("UrbanCart Sales Report\n")
    f.write("North: 1,250,000\n")
    f.write("South: 980,000\n")

with open("sample_report.txt", "r") as f:
    content = f.read()
print(content)

**Always use `with open(...) as f:`** rather than `f = open(...)` followed by a separate `f.close()`. The `with` block (a **context manager**) guarantees the file is properly closed even if an error occurs partway through reading or writing — a manually-called `.close()` after a crash never runs, silently leaking an open file handle. This is one of the very few "just always do it this way, no exceptions" rules in this entire course.

File modes you'll use: `"r"` (read, error if the file doesn't exist), `"w"` (write, **overwrites** the entire file if it exists — be careful), `"a"` (append to the end without erasing existing content).

### 11.2 CSV and JSON

In [ ]:
import csv, json

data = [["Region", "Sales"], ["North", 1250000], ["South", 980000]]
with open("temp_sales.csv", "w", newline="") as f:      # newline="" prevents extra blank rows on Windows
    writer = csv.writer(f)
    writer.writerows(data)

employee = {"name": "Priya", "role": "Analyst", "salary": 1250000}
with open("employee.json", "w") as f:
    json.dump(employee, f, indent=2)         # indent=2 makes the output human-readable

with open("employee.json") as f:
    loaded = json.load(f)
print(loaded)

**JSON and Python dictionaries map almost one-to-one** — `json.load()` turns a JSON file straight into a Python dict (or list of dicts for a JSON array), and `json.dump()` does the reverse. This matters enormously in practice: virtually every REST API you'll ever call from Python (for a business dashboard, an automated report, a data pull) returns JSON, and knowing this module means you already know how to work with that response.

💡 In practice, from Volume 02 onward, you'll almost always reach for `pd.read_csv()` and `pd.read_json()` instead of the raw `csv`/`json` modules, because Pandas gives you a full DataFrame with type inference, missing-value handling, and analysis methods for free. This chapter's raw versions matter because (a) you'll occasionally need fine-grained control Pandas doesn't expose, and (b) understanding what Pandas is doing underneath demystifies it.

### 11.3 Exception Handling

In [ ]:
try:
    result = 100 / 0
except ZeroDivisionError:
    print("Cannot divide by zero")
except Exception as e:               # a catch-all for anything not specifically handled above
    print("Error:", e)
else:
    print("Success")                 # runs ONLY if the try block raised no exception at all
finally:
    print("Cleanup always runs")     # runs NO MATTER WHAT — exception or not

The four-part `try / except / else / finally` structure is worth memorizing fully, not just the `try/except` part most tutorials stop at:

- **`try`**: the code that might fail.
- **`except SpecificError`**: catches *that particular* kind of failure. List the most specific exception types you anticipate (e.g., `FileNotFoundError`, `ZeroDivisionError`, `KeyError`) rather than always reaching for a blanket `except Exception` — specific handling lets you respond appropriately (retry, use a default, log and skip) instead of masking bugs you didn't anticipate.
- **`else`**: runs only if nothing went wrong — useful for code that should happen after a successful try, but that you don't want accidentally caught by the `except` block if *it* also fails.
- **`finally`**: always runs — the natural place to close a resource or print a "done" message regardless of outcome.

⚠️ A very common bad habit: wrapping large blocks of code in a bare `except:` (with no exception type at all) "just to be safe." This silently swallows *every* error, including genuine bugs like typos in variable names, making debugging far harder. Catch specific exceptions you actually expect and know how to handle.

### Cheat Sheet — Chapter 11

In [ ]:
with open(path, "r") as f:      # "r" read, "w" write (overwrites), "a" append
    content = f.read()

import csv
csv.writer(f).writerows(rows)
csv.reader(f)

import json
json.dump(obj, f, indent=2)
json.load(f)

try:
    risky_code()
except SpecificError:
    handle_it()
except Exception as e:
    print(e)
else:
    only_if_no_error()
finally:
    always_runs()

### 🎯 Business Challenge — Chapter 11

In [ ]:
import csv

# 1. Write a small CSV
rows = [["Region", "Sales"], ["North", 125000], ["South", 98000], ["East", 112000]]
with open("regional_sales.csv", "w", newline="") as f:
    csv.writer(f).writerows(rows)

# 2 & 3. Read it back with error handling
try:
    with open("regional_sales.csv", "r") as f:
        reader = csv.reader(f)
        header = next(reader)                       # skip and capture the header row
        total = sum(int(row[1]) for row in reader)  # sum the "Sales" column across remaining rows
    print(f"Total sales across regions: ₹{total:,}")
except FileNotFoundError:
    print("The sales file could not be found — check the path.")

---